In [1]:
import sys
print(sys.executable)

D:\Anaconda_file\envs\retailpulse\python.exe


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Load cleaned dataset
file_path = "../data/cleaned_transactions.csv"
df = pd.read_csv(file_path)

# Convert date column
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Cleaned dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Cleaned dataset loaded successfully!
Shape: (42750, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue']


In [3]:
# Set analysis date as one day after the last transaction
analysis_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

print("RFM analysis completed!")
print("Customers:", len(rfm))

display(rfm.head())

RFM analysis completed!
Customers: 1200


,Customer ID,Recency,Frequency,Monetary
0,10000.0,158,69,5658.65
1,10001.0,225,45,2675.70
2,10002.0,397,42,1700.28
3,10003.0,310,45,2157.64
4,10004.0,349,40,2421.04


In [4]:
# RFM scoring
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined RFM score
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

print("RFM scores created successfully!")

display(
    rfm[
        ["Customer ID", "Recency", "Frequency", "Monetary",
         "R_Score", "F_Score", "M_Score", "RFM_Score"]
    ].head(10)
)


RFM scores created successfully!


,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,10000.0,158,69,5658.65,5,5,5,555
1,10001.0,225,45,2675.70,5,4,5,545
2,10002.0,397,42,1700.28,3,4,4,344
3,10003.0,310,45,2157.64,5,4,4,544
4,10004.0,349,40,2421.04,4,4,4,444
5,10005.0,470,22,658.57,2,1,1,211
6,10006.0,264,51,3646.31,5,5,5,555
7,10007.0,462,26,786.79,2,2,2,222
8,10008.0,442,37,1420.14,3,3,3,333
9,10009.0,372,49,2162.16,4,5,4,454


In [5]:
# Create customer segments based on RFM scores

def assign_segment(row):
    if row["RFM_Score"] in ["555", "554", "545", "455", "454", "445"]:
        return "Champions"
    elif row["R_Score"] >= 4 and row["F_Score"] >= 3:
        return "Loyal Customers"
    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "Recent Customers"
    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    else:
        return "Needs Attention"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

print("Customer segmentation completed!")
print("\nSegment distribution:")
print(rfm["Segment"].value_counts())

display(rfm.head(10))

Customer segmentation completed!

Segment distribution:
Segment
Needs Attention     641
Champions           264
Loyal Customers     208
At Risk              78
Recent Customers      9
Name: count, dtype: int64


,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,10000.0,158,69,5658.65,5,5,5,555,Champions
1,10001.0,225,45,2675.70,5,4,5,545,Champions
2,10002.0,397,42,1700.28,3,4,4,344,Needs Attention
3,10003.0,310,45,2157.64,5,4,4,544,Loyal Customers
4,10004.0,349,40,2421.04,4,4,4,444,Loyal Customers
5,10005.0,470,22,658.57,2,1,1,211,Needs Attention
6,10006.0,264,51,3646.31,5,5,5,555,Champions
7,10007.0,462,26,786.79,2,2,2,222,Needs Attention
8,10008.0,442,37,1420.14,3,3,3,333,Needs Attention
9,10009.0,372,49,2162.16,4,5,4,454,Champions


In [6]:
rfm_path = "../data/rfm_customer_segments.csv"

rfm.to_csv(rfm_path, index=False)

print("RFM customer segmentation saved successfully!")
print("File:", rfm_path)
print("Customers:", len(rfm))

RFM customer segmentation saved successfully!
File: ../data/rfm_customer_segments.csv
Customers: 1200
